# Entity Resolution — Stage 1: Preprocessing Runner (Google Colab)

This notebook runs **Stage 1 (Preprocessing)** with separate handling for **Training** (`data/raw/train/`) and **Testing** (`data/raw/test/`) datasets, and automatically saves outputs to Google Drive under `data/processed/train/` and `data/processed/test/`.

### Stage 1 Core Guarantees:
1. **Unicode NFKD Normalization**
2. **Case Standardization** (Unicode-aware lowercase)
3. **Punctuation & Separator Normalization** (converted to spaces to preserve token boundaries)
4. **Whitespace Normalization**
5. **Non-destructive Tokenization** (stored as JSON arrays)
6. **Preservation of Numbers & Alphanumerics** (stored in `row_numbers` and tokens)
7. **Missing-Value Handling** (no `"nan"` / `"null"` artifacts)
8. **Row Preservation & Schema Validation**

--- 
## 1. Clone Repository & Install Dependencies

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Blackdevil-com/Wired-Mortus---Amazon-ML.git"
REPO_NAME = "Wired-Mortus---Amazon-ML"

# Clone if not present, then enter directory and pull latest code
if not Path("src/preprocessing").exists():
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}

!git fetch origin main
!git reset --hard origin/main

print(f"\nCurrent working directory: {Path.cwd()}")
!pip install -r requirements-colab.txt

--- 
## 2. Mount Google Drive & Setup Train / Test Folders
Links datasets from:
- `data/raw/train` $\rightarrow$ `data/processed/train`
- `data/raw/test` $\rightarrow$ `data/processed/test`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Base Google Drive directory
DRIVE_BASE = Path('/content/drive/MyDrive/entity-resolution')

# Drive folders for raw and processed datasets
DRIVE_RAW_TRAIN = DRIVE_BASE / 'data/raw/train'
DRIVE_RAW_TEST = DRIVE_BASE / 'data/raw/test'
DRIVE_PROC_TRAIN = DRIVE_BASE / 'data/processed/train'
DRIVE_PROC_TEST = DRIVE_BASE / 'data/processed/test'

# Create local and Drive directories
!mkdir -p data/raw/train data/raw/test data/processed/train data/processed/test
DRIVE_PROC_TRAIN.mkdir(parents=True, exist_ok=True)
DRIVE_PROC_TEST.mkdir(parents=True, exist_ok=True)

# Copy raw train datasets from Google Drive to local fast NVMe disk
if DRIVE_RAW_TRAIN.exists():
    print(f"Copying raw train datasets from {DRIVE_RAW_TRAIN}...")
    !cp -n "{DRIVE_RAW_TRAIN}"/*.tsv data/raw/train/ 2>/dev/null || true

# Copy raw test datasets from Google Drive to local fast NVMe disk
if DRIVE_RAW_TEST.exists():
    print(f"Copying raw test datasets from {DRIVE_RAW_TEST}...")
    !cp -n "{DRIVE_RAW_TEST}"/*.tsv data/raw/test/ 2>/dev/null || true

print("\n--- Local Raw Datasets Detected ---")
print("Train folder (data/raw/train):")
for f in sorted(Path("data/raw/train").glob("*.tsv")):
    print(f"  - {f.name} ({f.stat().st_size / (1024 * 1024):.2f} MB)")

print("Test folder (data/raw/test):")
for f in sorted(Path("data/raw/test").glob("*.tsv")):
    print(f"  - {f.name} ({f.stat().st_size / (1024 * 1024):.2f} MB)")

--- 
## 3. Run Pytest Suite
Run 111 unit & integration tests to verify normalization correctness before running on large datasets.

In [ ]:
!pytest tests/ -v

--- 
## 4. Preprocess TRAINING Data & Upload to Google Drive
Processes all TSVs in `data/raw/train/` $\rightarrow$ `data/processed/train/` and copies them to `DRIVE_BASE/data/processed/train/`.

In [ ]:
# Preprocess Training datasets
!python run_preprocessing.py --split train --raw-dir data/raw/train --processed-dir data/processed/train

# Upload preprocessed train files to Drive
print("\nUploading preprocessed train files to Google Drive...")
!cp -v data/processed/train/*.tsv "{DRIVE_PROC_TRAIN}/"

--- 
## 5. Preprocess TESTING Data & Upload to Google Drive
Processes all TSVs in `data/raw/test/` $\rightarrow$ `data/processed/test/` and copies them to `DRIVE_BASE/data/processed/test/`.

In [ ]:
# Preprocess Testing datasets
!python run_preprocessing.py --split test --raw-dir data/raw/test --processed-dir data/processed/test

# Upload preprocessed test files to Drive
print("\nUploading preprocessed test files to Google Drive...")
!cp -v data/processed/test/*.tsv "{DRIVE_PROC_TEST}/"

--- 
## 6. Inspect Sample Preprocessed Outputs
Display sample records from both train and test preprocessed files.

In [ ]:
import pandas as pd

proc_files = sorted(list(Path("data/processed").glob("**/*_preprocessed.tsv")))
print(f"Preprocessed files available ({len(proc_files)}):\n")

for pfile in proc_files:
    df = pd.read_csv(pfile, sep="\t", nrows=3, dtype=str, keep_default_na=False)
    print("=" * 75)
    print(f"File: {pfile.relative_to('data/processed')} | Records preview:")
    print("=" * 75)
    cols = [c for c in ['entity_id', 'business_name', 'business_name_normalized', 'business_name_tokens', 'business_address_normalized', 'row_numbers'] if c in df.columns]
    display(df[cols])